# Notebook 3: Classical Forecasting Models

## Overview

This notebook estimates ARIMA and Exponential Smoothing (ETS) models for each of the 10 MENA country series. Both models produce out-of-sample point forecasts and 95% prediction intervals for the 2019–2023 test period.

**ARIMA**: Order (p, d, q) selected by minimising AIC over a (p, d, q) ≤ 3 grid.  
**ETS**: Error/trend/seasonal specification selected automatically by AIC across additive, multiplicative, and damped-trend variants.

---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from src.data_processing import (
    MENA_COUNTRIES, get_country_series, train_test_split_series
)
from src.models import (
    select_arima_order, fit_arima, forecast_arima,
    fit_ets, forecast_ets,
)
from src.evaluation import compute_all_metrics
from src.visualization import plot_forecast

panel = pd.read_csv('../data/processed/mena_gdp_panel.csv')
TEST_YEARS = 5

## ARIMA Estimation

For each country, the AIC-optimal order is identified from the grid search and the model is fitted on the 1990–2018 training period. Clustered standard errors are not applicable here (univariate setting); standard ML-based inference is used.


In [ ]:
arima_results = {}
arima_forecasts = {}
arima_metrics = {}
arima_orders = {}

for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)

    best_order = select_arima_order(train)
    arima_orders[code] = best_order

    result = fit_arima(train, order=best_order)
    arima_results[code] = result

    fdf = forecast_arima(result, steps=len(test))
    arima_forecasts[code] = fdf

    metrics = compute_all_metrics(
        test.values, fdf['forecast'].values, train.values
    )
    arima_metrics[code] = metrics
    print(f'{name:20s}  order={best_order}  MAE={metrics["MAE"]:.2f}%  RMSE={metrics["RMSE"]:.2f}%')

In [ ]:
orders_df = pd.DataFrame.from_dict(arima_orders, orient='index', columns=['p', 'd', 'q'])
orders_df.index.name = 'Country'
orders_df.to_csv('../results/tables/arima_orders.csv')
print('ARIMA order selection:')
print(orders_df)

## ARIMA Forecasts

Forecast plots for each country show the training series, the actual test-period values, the point forecast, and the 95% prediction interval.


In [ ]:
for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)
    plot_forecast(
        train, test, arima_forecasts[code],
        model_name='ARIMA', country_name=name,
        save_path=f'../results/figures/03_arima_forecast_{code}.png',
    )

## Exponential Smoothing (ETS) Estimation

The Holt-Winters Exponential Smoothing model is fitted with automatic component selection (additive vs multiplicative trend, damped vs undamped) using AIC. Prediction intervals are constructed via a residual bootstrap (1,000 replications).


In [ ]:
ets_results = {}
ets_forecasts = {}
ets_metrics = {}

for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)

    fit = fit_ets(train)
    ets_results[code] = fit

    fdf = forecast_ets(fit, steps=len(test), series=train)
    ets_forecasts[code] = fdf

    metrics = compute_all_metrics(
        test.values, fdf['forecast'].values, train.values
    )
    ets_metrics[code] = metrics
    print(f'{name:20s}  MAE={metrics["MAE"]:.2f}%  RMSE={metrics["RMSE"]:.2f}%')

In [ ]:
for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)
    plot_forecast(
        train, test, ets_forecasts[code],
        model_name='ETS', country_name=name,
        save_path=f'../results/figures/03_ets_forecast_{code}.png',
    )

## Save Classical Model Results


In [ ]:
import json

for code in MENA_COUNTRIES:
    arima_forecasts[code].to_csv(f'../results/tables/arima_forecast_{code}.csv', index=False)
    ets_forecasts[code].to_csv(f'../results/tables/ets_forecast_{code}.csv', index=False)

pd.DataFrame(arima_metrics).T.to_csv('../results/tables/arima_metrics.csv')
pd.DataFrame(ets_metrics).T.to_csv('../results/tables/ets_metrics.csv')

print('Classical model results saved.')

## Summary

ARIMA order selection identified predominantly low-order models (AR(1) or ARMA(1,1) with one degree of differencing), consistent with the stationarity diagnostics in Notebook 2. ETS with additive damped trend performed best for the majority of countries. Notebook 4 estimates the advanced models (Prophet and LSTM).
